# AIC 2026 — Image captioning ingestion · BLIP-2 OPT 2.7B COCO

**Input:** `aqpahm/aic2026-keyframes-transnetv2`  
**Output:** `aqpahm/aic2026-caption-blip2-opt-2.7b-coco`  
**Model:** `Salesforce/blip2-opt-2.7b-coco`  
**Artifact/video:** `captions.parquet`, `_CAPTION_SUCCESS.json`

Notebook nguồn chạy độc lập trên Colab/Kaggle. Mỗi keyframe có **một caption tiếng Anh**. Không tự dịch sang tiếng Việt, không giả lập confidence và không lập search index. BLIP-2 chỉ nhìn từng ảnh; kết quả không thay thế OCR, ASR hoặc phân tích sự kiện theo thời gian.

**Trước khi chạy:** bật GPU và Internet; thêm `HF_TOKEN` vào Secrets, cho phép notebook truy cập. Token cần quyền đọc input và ghi output. Mặc định `MAX_VIDEOS_PER_RUN = 5` để pilot; xem vài caption ở cell QA, sau đó đổi thành `None` và chạy lại toàn bộ. Video có marker hợp lệ sẽ được skip. Một T4 Colab chạy một worker; hai T4 Kaggle chạy hai worker, mỗi GPU giữ một model riêng. Checkpoint khá lớn; lần tải đầu có thể lâu. Notebook kiểm tra caption thử trước khi xử lý video; nếu bước này lỗi thì không tạo output.

**Nếu từng chạy bản notebook cũ và gặp lỗi `PIL._typing._Ink`:** trong Colab chọn Runtime → Disconnect and delete runtime, kết nối lại rồi chạy notebook bản mới từ đầu. Cell cài đặt không còn nâng cấp Pillow vì thay thư viện ảnh giữa phiên có thể khiến các module Pillow trong bộ nhớ và trên đĩa lệch nhau.

Model card: https://huggingface.co/Salesforce/blip2-opt-2.7b-coco


In [ ]:
%pip install -q "huggingface_hub>=0.34,<2" "transformers==4.57.1" "accelerate>=1.0,<2" "safetensors>=0.4" "pyarrow>=17,<22" "pandas>=2.2,<3"


## 1. Cấu hình và runtime

Chỉ đổi cấu hình ở cell này. Nếu thay model, source revision, prompt hoặc generation settings thì output cũ không còn cùng contract; dùng dataset/version khác hoặc xử lý lại. `STRICT_REMOTE_VALIDATION=True` kiểm tra nội dung marker trước khi skip.


In [ ]:
import os
import sys
import tempfile

# Set Hub environment before importing huggingface_hub.
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "180"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "45"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import hashlib
import io
import json
import re
import shutil
import tarfile
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from queue import Queue

import pandas as pd
import torch
from IPython.display import display
try:
    import PIL
    from PIL import Image, ImageFile
except ImportError as exc:
    raise RuntimeError(
        'Pillow trong runtime khong dong bo. Colab: Runtime > Disconnect and delete runtime; '
        'sau do ket noi lai va Run all notebook moi.'
    ) from exc
print('Pillow:', PIL.__version__)
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download
from transformers import Blip2ForConditionalGeneration, Blip2Processor

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ---- Central configuration ----
INPUT_REPO = "aqpahm/aic2026-keyframes-transnetv2"
OUTPUT_REPO = "aqpahm/aic2026-caption-blip2-opt-2.7b-coco"
MODEL_ID = "Salesforce/blip2-opt-2.7b-coco"
PINNED_INPUT_REVISION = None  # Set a specific commit SHA for a reproducible rerun.
PINNED_MODEL_REVISION = None  # Set a specific commit SHA for a reproducible rerun.
OUTPUT_PRIVATE = True
MAX_VIDEOS_PER_RUN = 5  # Pilot. Set None for all actual videos after QA.
SELECT_LEVELS = None  # e.g. {"L21"}; None means all available L21–L30 videos.
INITIAL_BATCH_SIZE_T4 = 2
MAX_NEW_TOKENS = 40
NUM_BEAMS = 3
UPLOAD_BATCH_VIDEOS = 12
STRICT_REMOTE_VALIDATION = True
SCHEMA_VERSION = 1


def detect_runtime():
    if "google.colab" in sys.modules:
        return "colab"
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    return "local"


def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    if RUNTIME == "colab":
        from google.colab import userdata
        value = userdata.get(name)
    elif RUNTIME == "kaggle":
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    if not value:
        raise RuntimeError(f"Missing {name}. Add it to Colab/Kaggle Secrets and grant access.")
    return value


def work_root(name):
    if RUNTIME == "colab":
        return Path("/content") / name
    if RUNTIME == "kaggle":
        return Path("/kaggle/temp") / name
    return Path(tempfile.gettempdir()) / name


RUNTIME = detect_runtime()
ROOT = work_root("aic-caption-blip2")
DOWNLOAD_ROOT = ROOT / "downloads"
OUTPUT_ROOT = ROOT / "output"
for directory in (DOWNLOAD_ROOT, OUTPUT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Colab/Kaggle before running BLIP-2")

GPU_IDS = list(range(torch.cuda.device_count()))
DEVICES = [torch.device(f"cuda:{index}") for index in GPU_IDS]
BATCH_SIZE_BY_GPU = {}
for index in GPU_IDS:
    properties = torch.cuda.get_device_properties(index)
    vram_gib = properties.total_memory / 1024**3
    if vram_gib < 11:
        raise RuntimeError(f"cuda:{index} has only {vram_gib:.1f} GiB VRAM; this FP16 job expects at least 11 GiB")
    BATCH_SIZE_BY_GPU[index] = INITIAL_BATCH_SIZE_T4 if vram_gib < 20 else 4
    print(f"cuda:{index}: {properties.name}, {vram_gib:.1f} GiB, initial batch={BATCH_SIZE_BY_GPU[index]}")

HF_TOKEN = read_secret("HF_TOKEN")
api = HfApi(token=HF_TOKEN)
account = api.whoami()
print("Runtime:", RUNTIME, "| HF account:", account["name"])
api.create_repo(repo_id=OUTPUT_REPO, repo_type="dataset", private=OUTPUT_PRIVATE, exist_ok=True)
INPUT_REVISION = PINNED_INPUT_REVISION or api.dataset_info(INPUT_REPO).sha
MODEL_REVISION = PINNED_MODEL_REVISION or api.model_info(MODEL_ID).sha
print("Input revision:", INPUT_REVISION)
print("Model revision:", MODEL_REVISION)
print("Output:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")
print("GPU workers:", len(GPU_IDS))
torch.backends.cudnn.benchmark = True


## 2. Tải model và kiểm tra inference

Mỗi GPU nạp một bản FP16 riêng để xử lý song song. Không dùng `device_map="auto"` vì chế độ đó có thể chia một model qua hai GPU, làm mất hai worker độc lập. Lần đầu tải model sẽ tốn dung lượng mạng/đĩa đáng kể.


In [ ]:
processor = Blip2Processor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)


def configure_image_tokens(model):
    # Older BLIP-2 checkpoints have no image_token_index. Transformers 4.57
    # needs the processor to add image placeholders and the model to know their ID.
    query_count = model.config.num_query_tokens
    if not isinstance(query_count, int) or query_count <= 0:
        raise RuntimeError(f"Invalid BLIP-2 num_query_tokens: {query_count!r}")
    processor.num_query_tokens = query_count
    image_token = processor.image_token.content
    image_token_id = processor.tokenizer.convert_tokens_to_ids(image_token)
    if not image_token or not isinstance(image_token_id, int) or image_token_id < 0:
        raise RuntimeError("BLIP-2 processor has no valid image token")
    if len(processor.tokenizer) > model.get_input_embeddings().num_embeddings:
        model.resize_token_embeddings(len(processor.tokenizer))
    if image_token_id >= model.get_input_embeddings().num_embeddings:
        raise RuntimeError("BLIP-2 image token ID exceeds model vocabulary")
    model.config.image_token_index = image_token_id
    return query_count, image_token_id


models = []
for device_id in GPU_IDS:
    device = DEVICES[device_id]
    print(f"Loading {MODEL_ID} on {device}...")
    model = Blip2ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        use_safetensors=True,
        device_map={"": str(device)},
    )
    query_count, image_token_id = configure_image_tokens(model)
    model.eval()
    models.append(model)
    print(
        f"Loaded on {device}; image token={image_token_id}, queries={query_count}, "
        f"VRAM allocated={torch.cuda.memory_allocated(device_id) / 1024**3:.2f} GiB"
    )
    gc.collect()


def generate_captions(images, model, device):
    # Even unconditional captioning needs explicit image-token placeholders.
    batch = processor(images=images, text=[""] * len(images), return_tensors="pt")
    expected = model.config.num_query_tokens
    actual = (batch["input_ids"] == model.config.image_token_index).sum(dim=1)
    if not torch.all(actual == expected):
        raise RuntimeError(f"BLIP-2 image placeholders mismatch: {actual.tolist()} vs {expected}")
    pixel_values = batch["pixel_values"].to(device=device, dtype=torch.float16, non_blocking=True)
    input_ids = batch["input_ids"].to(device=device, non_blocking=True)
    attention_mask = batch["attention_mask"].to(device=device, non_blocking=True)
    with torch.inference_mode():
        generated = model.generate(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            do_sample=False,
            use_cache=True,
        )
    texts = [" ".join(text.split()) for text in processor.batch_decode(generated, skip_special_tokens=True)]
    if len(texts) != len(images):
        raise RuntimeError(f"Caption count mismatch: {len(texts)} vs {len(images)}")
    if any(not text for text in texts):
        raise RuntimeError("BLIP-2 generated an empty caption; no success marker will be published")
    return texts


for index, model in enumerate(models):
    dummy = Image.new("RGB", (364, 364), color=(90, 110, 120))
    try:
        smoke = generate_captions([dummy], model, DEVICES[index])
        print(f"cuda:{index} smoke test:", smoke[0][:120])
    finally:
        dummy.close()
        with torch.cuda.device(index):
            torch.cuda.empty_cache()


## 3. Đọc keyframe và tạo caption

Caption giữ cùng `frame_uid`, `frame_idx`, `sample_n`, `shot_id` và `timestamp_sec` với dataset keyframe. Không coi caption là OCR; BLIP-2 có thể bỏ qua chữ nhỏ hoặc đoán sai tên riêng.


In [ ]:
def retry(operation, description, attempts=7):
    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except Exception as error:
            if attempt == attempts:
                raise
            delay = min(300, 5 * 2 ** (attempt - 1))
            print(f"{description}: {type(error).__name__}; retry in {delay}s [{attempt}/{attempts}]")
            time.sleep(delay)


def download_input(filename, local_dir):
    local_dir.mkdir(parents=True, exist_ok=True)
    return Path(retry(
        lambda: hf_hub_download(
            repo_id=INPUT_REPO,
            repo_type="dataset",
            filename=filename,
            revision=INPUT_REVISION,
            token=HF_TOKEN,
            local_dir=local_dir,
        ),
        f"download {filename}",
    ))


def normalized_tar_name(value):
    return PurePosixPath(str(value).replace("\\", "/")).as_posix().lstrip("./")


def build_tar_lookup(archive):
    by_name = {}
    by_basename = defaultdict(list)
    for member in archive.getmembers():
        if member.isfile():
            name = normalized_tar_name(member.name)
            by_name[name] = member
            by_basename[PurePosixPath(name).name].append(member)
    return by_name, by_basename


def locate_member(image_path, by_name, by_basename):
    expected = normalized_tar_name(image_path)
    if expected in by_name:
        return by_name[expected]
    matches = by_basename.get(PurePosixPath(expected).name, [])
    if len(matches) != 1:
        raise RuntimeError(f"Cannot uniquely find {image_path!r} inside keyframes.tar")
    return matches[0]


def read_images(archive, records, by_name, by_basename):
    images = []
    try:
        for record in records:
            member = locate_member(record["image_path"], by_name, by_basename)
            stream = archive.extractfile(member)
            if stream is None:
                raise RuntimeError(f"Cannot read {member.name}")
            with Image.open(io.BytesIO(stream.read())) as source:
                image = source.convert("RGB")
                image.load()
            images.append(image)
        return images
    except Exception:
        for image in images:
            image.close()
        raise


IDENTITY_COLUMNS = ["video_id", "frame_uid", "sample_n", "frame_idx", "shot_id", "timestamp_sec", "image_path"]


def make_manifest(source, video_id):
    required = {"video_id", "sample_n", "frame_idx", "shot_id", "timestamp_sec", "image_path"}
    missing = required - set(source.columns)
    if missing:
        raise RuntimeError(f"{video_id}: missing source columns {sorted(missing)}")
    if source.empty:
        raise RuntimeError(f"{video_id}: empty source metadata")
    frame = source.sort_values("sample_n", kind="stable").reset_index(drop=True).copy()
    if frame["video_id"].astype(str).nunique() != 1 or str(frame.iloc[0]["video_id"]) != video_id:
        raise RuntimeError(f"{video_id}: video_id mismatch")
    if frame["sample_n"].duplicated().any() or frame["frame_idx"].duplicated().any():
        raise RuntimeError(f"{video_id}: duplicated sample_n/frame_idx")
    if frame["image_path"].isna().any() or frame["timestamp_sec"].isna().any():
        raise RuntimeError(f"{video_id}: missing image_path/timestamp_sec")
    frame["video_id"] = frame["video_id"].astype(str)
    frame["sample_n"] = frame["sample_n"].astype("int64")
    frame["frame_idx"] = frame["frame_idx"].astype("int64")
    frame["shot_id"] = frame["shot_id"].astype("int64")
    frame["timestamp_sec"] = frame["timestamp_sec"].astype("float64")
    frame["image_path"] = frame["image_path"].astype(str)
    frame["frame_uid"] = [f"{video_id}:{number}" for number in frame["frame_idx"].tolist()]
    return frame[IDENTITY_COLUMNS]


def caption_video(archive, manifest, model, device, initial_batch):
    by_name, by_basename = build_tar_lookup(archive)
    records = manifest.to_dict("records")
    captions = []
    cursor = 0
    batch_size = min(initial_batch, len(records))
    minimum_batch = batch_size
    inference_sec = 0.0
    while cursor < len(records):
        selected = records[cursor:cursor + batch_size]
        images = read_images(archive, selected, by_name, by_basename)
        started = time.perf_counter()
        try:
            texts = generate_captions(images, model, device)
            torch.cuda.synchronize(device)
        except (torch.OutOfMemoryError, RuntimeError) as error:
            oom = isinstance(error, torch.OutOfMemoryError) or "out of memory" in str(error).lower()
            if not oom or batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            minimum_batch = min(minimum_batch, batch_size)
            print(f"{device}: CUDA OOM; reducing batch to {batch_size}")
            gc.collect()
            with torch.cuda.device(device):
                torch.cuda.empty_cache()
            continue
        finally:
            for image in images:
                image.close()
        inference_sec += time.perf_counter() - started
        captions.extend(texts)
        cursor += len(selected)
        if cursor % 128 < len(selected) or cursor == len(records):
            print(f"  captioned {cursor}/{len(records)}")
    if len(captions) != len(manifest):
        raise RuntimeError("Caption count does not match source frame count")
    return captions, minimum_batch, inference_sec


## 4. Kiểm tra từ xa và lập danh sách video cần chạy

Chỉ các video thực tế có TAR, Parquet và keyframe success marker mới được nhận. Chế độ strict đọc marker caption từ xa và đối chiếu model, source revision, schema và generation settings trước khi skip.


In [ ]:
input_files = set(retry(
    lambda: api.list_repo_files(INPUT_REPO, repo_type="dataset", revision=INPUT_REVISION),
    "list input files",
))
output_files = set(retry(
    lambda: api.list_repo_files(OUTPUT_REPO, repo_type="dataset"),
    "list output files",
))

tar_pattern = re.compile(r"^data/(L(?:2[1-9]|30))/(L\d+_V\d+)/keyframes\.tar$")
video_specs = []
for filename in sorted(input_files):
    match = tar_pattern.fullmatch(filename)
    if not match:
        continue
    level, video_id = match.groups()
    if SELECT_LEVELS is not None and level not in SELECT_LEVELS:
        continue
    prefix = f"data/{level}/{video_id}"
    frames_filename = f"{prefix}/frames.parquet"
    source_marker = f"{prefix}/_SUCCESS.json"
    if frames_filename not in input_files or source_marker not in input_files:
        raise RuntimeError(f"Incomplete keyframe source for {video_id}")
    video_specs.append({
        "level": level,
        "video_id": video_id,
        "tar_filename": filename,
        "frames_filename": frames_filename,
    })


def remote_complete(spec, files):
    prefix = f"data/{spec['level']}/{spec['video_id']}"
    required = {f"{prefix}/captions.parquet", f"{prefix}/_CAPTION_SUCCESS.json"}
    if not required.issubset(files):
        return False
    if not STRICT_REMOTE_VALIDATION:
        return True
    try:
        marker_path = retry(
            lambda: hf_hub_download(
                repo_id=OUTPUT_REPO,
                repo_type="dataset",
                filename=f"{prefix}/_CAPTION_SUCCESS.json",
                token=HF_TOKEN,
            ),
            f"validate marker {spec['video_id']}",
        )
        marker = json.loads(Path(marker_path).read_text(encoding="utf-8"))
        return (
            marker.get("status") == "success"
            and marker.get("video_id") == spec["video_id"]
            and marker.get("source_repo") == INPUT_REPO
            and marker.get("source_revision") == INPUT_REVISION
            and marker.get("model_id") == MODEL_ID
            and marker.get("model_revision") == MODEL_REVISION
            and marker.get("language") == "en"
            and marker.get("caption_policy") == "all-keyframes"
            and int(marker.get("frames", 0)) > 0
            and int(marker.get("captions", 0)) == int(marker.get("frames", 0))
            and int(marker.get("max_new_tokens", 0)) == MAX_NEW_TOKENS
            and int(marker.get("num_beams", 0)) == NUM_BEAMS
            and int(marker.get("schema_version", 0)) == SCHEMA_VERSION
        )
    except Exception as error:
        print(f"Invalid/unavailable marker for {spec['video_id']}: {error!r}")
        return False


if not video_specs:
    raise RuntimeError(f"No complete keyframe videos found in {INPUT_REPO}")

print("Validating existing remote output...")
with ThreadPoolExecutor(max_workers=12) as validator:
    flags = list(validator.map(lambda spec: remote_complete(spec, output_files), video_specs))
completed_specs = [spec for spec, okay in zip(video_specs, flags) if okay]
pending_specs = [spec for spec, okay in zip(video_specs, flags) if not okay]
if MAX_VIDEOS_PER_RUN is not None:
    pending_specs = pending_specs[:MAX_VIDEOS_PER_RUN]
print("Source videos:", len(video_specs))
print("Already complete:", len(completed_specs))
print("Selected this run:", len(pending_specs))


## 5. Chạy ingest, upload theo lô và resume

Upload `captions.parquet` và marker trong **cùng một commit**, nên marker không thể xuất hiện trước dữ liệu. Khi phiên ngắt, chạy lại từ đầu; notebook kiểm tra output từ xa và tiếp tục các video chưa hoàn chỉnh.


In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def catalog_sha256(manifest):
    digest = hashlib.sha256()
    for row in manifest.itertuples(index=False):
        payload = {
            "video_id": row.video_id,
            "frame_uid": row.frame_uid,
            "sample_n": int(row.sample_n),
            "frame_idx": int(row.frame_idx),
            "shot_id": int(row.shot_id),
            "timestamp_sec": float(row.timestamp_sec),
            "image_path": row.image_path,
        }
        digest.update(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


def write_output(spec, manifest, captions, stats):
    directory = OUTPUT_ROOT / spec["level"] / spec["video_id"]
    shutil.rmtree(directory, ignore_errors=True)
    directory.mkdir(parents=True, exist_ok=True)
    parquet_path = directory / "captions.parquet"
    marker_path = directory / "_CAPTION_SUCCESS.json"

    frame = manifest.copy()
    frame["caption_en"] = captions
    frame["caption_en_normalized"] = [" ".join(text.lower().split()) for text in captions]
    frame["language"] = "en"
    frame["model_id"] = MODEL_ID
    frame["model_revision"] = MODEL_REVISION
    frame.to_parquet(parquet_path, index=False)

    # Read back before publishing a success marker.
    stored = pd.read_parquet(parquet_path)
    if len(stored) != len(manifest):
        raise RuntimeError(f"Stored row count mismatch for {spec['video_id']}")
    if stored["frame_uid"].tolist() != manifest["frame_uid"].tolist():
        raise RuntimeError(f"Stored frame ordering mismatch for {spec['video_id']}")
    if stored["caption_en"].isna().any() or (stored["caption_en"].str.len() == 0).any():
        raise RuntimeError(f"Stored empty captions for {spec['video_id']}")

    marker = {
        "video_id": spec["video_id"],
        "status": "success",
        "frames": len(manifest),
        "captions": len(captions),
        "language": "en",
        "caption_policy": "all-keyframes",
        "source_repo": INPUT_REPO,
        "source_revision": INPUT_REVISION,
        "source_tar": spec["tar_filename"],
        "source_frames": spec["frames_filename"],
        "catalog_sha256": catalog_sha256(manifest),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "processor_id": MODEL_ID,
        "generation_mode": "deterministic-beam-search",
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_beams": NUM_BEAMS,
        "parquet_sha256": sha256_file(parquet_path),
        "stats": stats,
        "schema_version": SCHEMA_VERSION,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    marker_path.write_text(json.dumps(marker, ensure_ascii=False, indent=2), encoding="utf-8")
    return [parquet_path, marker_path]


device_pool = Queue()
for model_index in range(len(models)):
    device_pool.put(model_index)


def process_video(spec):
    model_index = device_pool.get()
    device_id = GPU_IDS[model_index]
    device = DEVICES[model_index]
    video_id = spec["video_id"]
    task_dir = DOWNLOAD_ROOT / video_id
    shutil.rmtree(task_dir, ignore_errors=True)
    task_dir.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(device_id)
    try:
        frames_path = download_input(spec["frames_filename"], task_dir)
        tar_path = download_input(spec["tar_filename"], task_dir)
        manifest = make_manifest(pd.read_parquet(frames_path), video_id)
        with tarfile.open(tar_path, mode="r:*") as archive:
            captions, minimum_batch, inference_sec = caption_video(
                archive, manifest, models[model_index], device, BATCH_SIZE_BY_GPU[device_id]
            )
        torch.cuda.synchronize(device)
        elapsed = time.perf_counter() - started
        stats = {
            "elapsed_sec": round(elapsed, 3),
            "inference_sec": round(inference_sec, 3),
            "inference_fps": round(len(captions) / max(inference_sec, 1e-6), 3),
            "peak_vram_gib": round(torch.cuda.max_memory_allocated(device_id) / 1024**3, 3),
            "minimum_batch_size": minimum_batch,
            "device": f"cuda:{device_id}",
            "gpu": torch.cuda.get_device_name(device_id),
        }
        paths = write_output(spec, manifest, captions, stats)
        return {"video_id": video_id, "paths": paths, "frames": len(manifest), "stats": stats}
    finally:
        shutil.rmtree(task_dir, ignore_errors=True)
        gc.collect()
        with torch.cuda.device(device_id):
            torch.cuda.empty_cache()
        device_pool.put(model_index)


def upload_outputs(paths, video_ids):
    operations = [CommitOperationAdd(
        path_in_repo=f"data/{path.relative_to(OUTPUT_ROOT).as_posix()}",
        path_or_fileobj=str(path),
    ) for path in paths]
    def commit():
        return api.create_commit(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            operations=operations,
            commit_message=f"Add BLIP-2 captions for {video_ids[0]} through {video_ids[-1]}",
        )
    try:
        retry(commit, f"upload {len(video_ids)} videos")
    except Exception:
        # A timeout can occur after the server accepted the atomic commit.
        current_files = set(retry(
            lambda: api.list_repo_files(OUTPUT_REPO, repo_type="dataset"),
            "check ambiguous upload",
        ))
        specs = [{"level": video_id.split("_")[0], "video_id": video_id} for video_id in video_ids]
        if not all(remote_complete(spec, current_files) for spec in specs):
            raise
        print("Commit had an ambiguous response; all remote markers are valid")


pending_paths = []
pending_ids = []
run_results = []
failures = []


def flush_pending():
    global pending_paths, pending_ids
    if not pending_paths:
        return
    upload_outputs(pending_paths, pending_ids)
    print(f"Uploaded {len(pending_ids)} videos in one commit")
    for video_id in pending_ids:
        shutil.rmtree(OUTPUT_ROOT / video_id.split("_")[0] / video_id, ignore_errors=True)
    pending_paths = []
    pending_ids = []


run_started = time.perf_counter()
with ThreadPoolExecutor(max_workers=len(models)) as executor:
    futures = {executor.submit(process_video, spec): spec for spec in pending_specs}
    for position, future in enumerate(as_completed(futures), 1):
        spec = futures[future]
        try:
            result = future.result()
            run_results.append(result)
            pending_paths.extend(result["paths"])
            pending_ids.append(result["video_id"])
            stats = result["stats"]
            print(
                f"[{position}/{len(futures)}] OK {result['video_id']}: "
                f"{result['frames']} captions, {stats['inference_fps']} inference FPS, "
                f"{stats['peak_vram_gib']} GiB, {stats['device']}"
            )
            if len(pending_ids) >= UPLOAD_BATCH_VIDEOS:
                flush_pending()
        except Exception as error:
            failures.append({"video_id": spec["video_id"], "error": repr(error)})
            print(f"[{position}/{len(futures)}] FAILED {spec['video_id']}: {error!r}")

flush_pending()
print("=" * 72)
print("GPU workers:", len(models))
print("Processed this run:", len(run_results))
print("Failures:", len(failures))
if run_results:
    total = sum(item["frames"] for item in run_results)
    print("Captions:", total)
    print("Approximate parallel FPS:", round(total / max(time.perf_counter() - run_started, 1e-6), 3))
if failures:
    print(json.dumps(failures, ensure_ascii=False, indent=2))
    raise RuntimeError("Caption run has failures. Rerun; valid completed videos will be skipped.")


## 6. Kiểm tra kết quả và audit

Cell này kiểm tra số video thành công trên Hugging Face. Pilot sẽ báo các video còn lại mà không xem đó là lỗi. Mở Parquet của vài video và xem `caption_en` so với ảnh, nhất là cảnh chữ Việt, bản đồ, ảnh mờ và các sự kiện không thể suy ra từ một frame.


In [ ]:
final_files = set(retry(
    lambda: api.list_repo_files(OUTPUT_REPO, repo_type="dataset"),
    "refresh output files",
))
with ThreadPoolExecutor(max_workers=12) as validator:
    final_flags = list(validator.map(lambda spec: remote_complete(spec, final_files), video_specs))
missing = [spec["video_id"] for spec, okay in zip(video_specs, final_flags) if not okay]

print(f"Remote complete: {len(video_specs) - len(missing)}/{len(video_specs)} videos")
print("Dataset:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")
if missing:
    print("Remaining:", len(missing), "sample:", missing[:20])
if MAX_VIDEOS_PER_RUN is None and missing:
    raise RuntimeError(f"Missing valid caption output for {len(missing)} videos")
if MAX_VIDEOS_PER_RUN is not None:
    print("Pilot finished. Inspect captions, set MAX_VIDEOS_PER_RUN=None and rerun all cells.")

for result in run_results[:2]:
    level = result["video_id"].split("_")[0]
    filename = f"data/{level}/{result['video_id']}/captions.parquet"
    path = hf_hub_download(repo_id=OUTPUT_REPO, repo_type="dataset", filename=filename, token=HF_TOKEN)
    sample = pd.read_parquet(path, columns=["frame_uid", "timestamp_sec", "caption_en"]).head(5)
    print("\nQA sample:", result["video_id"])
    display(sample)
